Semantic Caching

In [ ]:
from datetime import datetime

In [ ]:
from EmbeddingFunction import MyEmbeddingFunction
embedding_fuction = MyEmbeddingFunction()


In [ ]:
embeddings = embedding_fuction.__call__(["Hello world", "This is a test sentence.", "I love programming in Python!"])
print(len(embeddings))

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./ChromaDB")

In [ ]:
if "my_collection" in [collection.name for collection in client.list_collections()]:
    client.delete_collection("my_collection")
collection = client.create_collection(
    "my_collection",
    embedding_function=embedding_fuction,
    configuration={
        "hnsw": {
            "space": "cosine"
        }
    }
    )

In [ ]:
collection.add(
    documents=["Hello world", "This is a test sentence.", "I love programming in Python."],
    metadatas=[{"language": "English"}, {"language": "English"}, {"language": "English"}],
    ids=["1", "2", "3"],
)

In [ ]:
result = collection.query(
    query_texts=["I love"],
    n_results=1
)

In [ ]:
print(result)

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)
open_router_api_key = os.getenv("OPEN_ROUTER_API_KEY")

if open_router_api_key is None:
    raise ValueError("OPEN_ROUTER_API_KEY environment variable is not set.")
else:
    print("OPEN_ROUTER_API_KEY environment variable is set.")

In [ ]:
#call llm
from openai import OpenAI
open_router = OpenAI(api_key=open_router_api_key, base_url="https://openrouter.ai/api/v1")

In [ ]:
SYSTEM_PROMPT = "your are an helpful assistant"

In [ ]:
# USER_PROMPT = "Explain about the pyton programming language"
USER_PROMPT = "Explain about interpreted language"
messages = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT}]

In [ ]:
embedding_id = 0

In [ ]:
def build_cache_item(user_prompt: str, embedding_id: int, response: str):
    cacheId = "cache-" + str(embedding_id)
    document = user_prompt
    metadata = {
        "timestamp": str(datetime.now()),
        "response": response,
    }
    return cacheId, document, metadata

In [ ]:
result = collection.query(
    query_texts=[USER_PROMPT],
    n_results=1
)

distance = result['distances'][0][0]
print("Similarity Distance:", distance)

if distance < 0.2:
    print("Semantically similar")
    print("Retrieving from cache")

    print("Cached Response:", result['metadatas'][0][0]['response'])
else:
    print("Not semantically similar")
    print("calling LLM")

    response = open_router.chat.completions.create(
        model="openrouter/free",
        messages=messages,
        temperature=0.5,
        max_tokens=200
    )

    response_text = response.choices[0].message.content
    print("LLM Response:", response_text)

    embedding_id += 1

    cacheId, document, metadata = build_cache_item(USER_PROMPT, embedding_id, response_text)

    collection.add(
        ids=[cacheId],
        documents=[document],
        metadatas=[metadata]
    )
    

In [ ]:
# Need to use the uuid for the embedding id generation
# Need to move the respone to redis with ttl policy